# Conversational RAG: chat with your documents over multiple turns

A chatbot over your own documents is just RAG in a loop, with two extra ingredients: **conversation memory** (so follow-up questions like "what is *its* capital?" resolve) and **per-turn retrieval** that grounds every answer in your data with citations.

This notebook builds that loop on [Infino](https://pypi.org/project/infino/) over real Wikipedia articles: ingest once into a local table, then hold a multi-turn conversation where each turn retrieves with **hybrid search** (BM25 + vector) and answers from the retrieved sources. One engine, on local disk — no separate vector database.

## Setup

In [1]:
# %pip install infino pyarrow sentence-transformers datasets

In [2]:
import sys
from pathlib import Path

# Make the repo's shared helpers importable.
sys.path.insert(0, str(Path.cwd().parent))

import shutil

import infino
import pyarrow as pa

from _shared.embedding import DIM, METRIC, embed, embed_query, as_vector_column
from _shared.chunking import chunk_text
from _shared.loaders import load_wikipedia
from _shared.sql import sql_lit, query
from _shared.llm import complete

## 1. Ingest documents (once, to local disk)

Load a sample of Wikipedia articles, chunk them, embed locally, and index `text` (BM25) + `emb` (vector) in one table. The index is durable on local disk, so it survives across sessions.

Wikipedia articles are long, so chunking matters more here than for short abstracts — tune `chunk_words` / `overlap_words` in `_shared/chunking.py` to fit your documents.

In [3]:
DB_DIR = "./wiki_chat_data"
shutil.rmtree(DB_DIR, ignore_errors=True)

articles = load_wikipedia(n=200)
titles, texts = [], []
for article in articles:
    for chunk in chunk_text(article["text"]):
        titles.append(article["title"])
        texts.append(chunk)
print(f"{len(articles)} articles -> {len(texts)} chunks")

db = infino.connect(DB_DIR)
schema = pa.schema([
    pa.field("title", pa.large_utf8(), nullable=False),
    pa.field("text", pa.large_utf8(), nullable=False),
    pa.field("emb", pa.list_(pa.float32(), DIM), nullable=False),
])
spec = infino.IndexSpec().fts("text").vector("emb", DIM, n_cent=128, metric=METRIC)
table = db.create_table("docs", schema, spec)
table.append(pa.record_batch([
    pa.array(titles, type=pa.large_utf8()),
    pa.array(texts, type=pa.large_utf8()),
    as_vector_column(embed(texts)),
], schema=schema))
print("indexed", db.query_sql("SELECT COUNT(*) AS n FROM docs").to_pydict()["n"][0], "chunks")

200 articles -> 1391 chunks


indexed 1391 chunks


## 2. Retrieval with conversation memory

Each turn retrieves with `hybrid_search` (BM25 + vector, fused in-engine). The trick for follow-ups: fold the **recent user turns into the search query** so a pronoun like "its" or "there" still points at the right entity.

In [4]:
def retrieve(search_query: str, k: int = 3) -> list[dict]:
    qvec = ",".join(str(x) for x in embed_query(search_query))
    sql = (
        f"SELECT p.title, p.text, h.score "
        f"FROM hybrid_search('docs', 'text', {sql_lit(search_query)}, 'emb', '{qvec}', {max(k, 10)}) h "
        f"JOIN docs p ON h._id = p._id ORDER BY h.score DESC LIMIT {k}"
    )
    cols = query(db, sql)
    return [
        {"title": t, "text": x, "score": s}
        for t, x, s in zip(cols.get("title", []), cols.get("text", []), cols.get("score", []))
    ]

## 3. Answer from sources

Build a prompt from the conversation so far plus the retrieved passages. `complete` (from `_shared.llm`) uses Azure AI Foundry or OpenAI if configured, and returns `None` otherwise — so the loop shows the grounded sources and still runs key-free.

In [5]:
def answer(question: str, history: list[str], hits: list[dict]) -> str | None:
    context = "\n".join(f"[{i + 1}] {h['title']}: {h['text']}" for i, h in enumerate(hits))
    convo = "\n".join(history)
    prompt = (
        "Continue the conversation. Answer the last question using only the "
        "context passages and cite them as [n]. If the answer isn't in the "
        "context, say so.\n\n"
        f"Conversation so far:\n{convo}\n\nContext:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    return complete(prompt)

## 4. Hold a multi-turn conversation

`chat` ties it together: fold history into the retrieval query, retrieve, answer, and remember the turn. Watch turn 2 — "its capital" resolves to the topic from turn 1, and retrieval surfaces the right passage.

In [6]:
history: list[str] = []


def chat(question: str, k: int = 3):
    # Fold the last couple of user turns in so follow-ups resolve.
    search_query = " ".join(history[-2:] + [question])
    hits = retrieve(search_query, k=k)
    reply = answer(question, history, hits)
    history.append(question)

    print(f"\nUSER: {question}")
    print(f"ASSISTANT: {reply}" if reply else "ASSISTANT (no LLM configured \u2014 grounded sources):")
    for i, h in enumerate(hits, 1):
        print(f"  [{i}] {h['title']}: {h['text'][:100]}")


chat("Tell me about Australia.")
chat("What is its capital city?")
chat("What languages are spoken there?")


USER: Tell me about Australia.
ASSISTANT: Australia is a country and sovereign state in the southern hemisphere, located in Oceania [1]. Its capital city is Canberra, and its largest city is Sydney [1][2]. Australia is the sixth biggest country in the world by land area [1].

It has six states—New South Wales, Queensland, South Australia, Victoria, Western Australia, and Tasmania—and two major mainland territories: the Northern Territory and the Australian Capital Territory [2]. Most Australians live in cities along the coast, including Sydney, Melbourne, Brisbane, Perth, Adelaide, Newcastle, and the Gold Coast [2].

Australia has about 25–26 million people [1][2]. It is associated with Australasia, which includes Australia, New Zealand, New Guinea, and nearby islands on the Australian tectonic plate [1]. Some sources also describe Australia as part of the larger region of Oceania, which includes Australia, New Zealand, and the Pacific Islands [1][3].
  [1] Australia: Australia (offic


USER: What is its capital city?
ASSISTANT: Its capital city is Canberra [2].
  [1] Australia: to carry out all the duties of the King in Australia. Regions and cities Australia has six states, t
  [2] Australia: Australia (officially called the Commonwealth of Australia) is a country and sovereign state in the 
  [3] Australia: Gold Coast. The largest inland city is Canberra, which is also the nation's capital. The largest cit



USER: What languages are spoken there?
ASSISTANT: English is the main spoken language in Australia [3].
  [1] Australia: to carry out all the duties of the King in Australia. Regions and cities Australia has six states, t
  [2] Australia: Australia (officially called the Commonwealth of Australia) is a country and sovereign state in the 
  [3] Australia: of Australia, who is the member of Parliament chosen as leader. The current Prime Minister is Anthon


## What just happened

A full chat-with-docs loop: ingest once, then every turn retrieves from one Infino table with hybrid search and answers from cited sources, with conversation memory folded into retrieval so follow-ups work. The table is durable on local disk and is simultaneously BM25-, vector-, and SQL-queryable — no separate vector store to run or sync.

To turn this into a live app, wrap `chat()` in a web UI (e.g. Streamlit) and stream the answer — the retrieval and grounding logic stays exactly as above.